In [ ]:
import pandas as pd
acc = pd.read_csv('acc_sgg.csv')
veh = pd.read_csv('veh_sgg_202603.csv')
pop = pd.read_csv('PopAge202509.csv', dtype={'행정기관코드': str})

print(sorted(acc['사고발생시도'].unique()))  # 사고 시도명
print(veh['사용본거지주소명'].str.split().str[0].value_counts())  # 차량 시도명
print(sorted(pop['시도명'].unique()))  # 인구 시도명

In [ ]:
sd = {'서울특별시':'서울','부산광역시':'부산','부산직할시':'부산','대구광역시':'대구','대구직할시':'대구',
      '인천광역시':'인천','인천직할시':'인천','광주광역시':'광주','대전광역시':'대전','울산광역시':'울산',
      '세종특별자치시':'세종','경기도':'경기','경기':'경기','강원도':'강원','강원특별자치도':'강원',
      '충청북도':'충북','충청남도':'충남','전라북도':'전북','전북특별자치도':'전북','전라남도':'전남',
      '경상북도':'경북','경상남도':'경남','제주특별자치도':'제주'}

a = veh['사용본거지주소명'].str.split()
veh['시도'] = a.str[0].map(sd); veh['시군구'] = a.str[1]          # 일반구는 시 단위로 묶임
pop['시도'] = pop['시도명'].map(sd); pop['시군구'] = pop['시군구명'].str.split().str[0]
acc['시도'] = acc['사고발생시도']; acc['시군구'] = acc['사고발생시군구'].str.split().str[0]

ka = set(zip(acc['시도'], acc['시군구'])); kv = set(zip(veh['시도'], veh['시군구'])); kp = set(zip(pop['시도'], pop['시군구']))
print(len(ka), len(kv), len(kp))
print('사고에만:', sorted(ka - kp))
print('인구에만:', sorted(kp - ka))
print('차량에만:', sorted(kv - kp, key=str)[:30])

In [ ]:
fix = {('경남','창원시(통합)'):('경남','창원시'), ('경북','군위군'):('대구','군위군'),
       ('세종','세종특별자치시'):('세종','세종시'), ('경기','양주'):('경기','양주시'),
       ('경기','포천'):('경기','포천시'), ('경기','여주군'):('경기','여주시'),
       ('인천','남구'):('인천','미추홀구'), ('인천','북구'):('인천','부평구'),
       ('충남','당진군'):('충남','당진시'), ('충남','온양시'):('충남','아산시'), ('충북','청원군'):('충북','청주시')}

def fx(df):
    k = [fix.get((a, b), (a, b)) for a, b in zip(df['시도'], df['시군구'])]
    df['시도'] = [x[0] for x in k]; df['시군구'] = [x[1] for x in k]
    return df

pop.loc[pop['시도'] == '세종', '시군구'] = '세종시'
acc, veh, pop = fx(acc), fx(veh), fx(pop)

ka = set(zip(acc['시도'], acc['시군구'])); kv = set(zip(veh['시도'], veh['시군구'])); kp = set(zip(pop['시도'], pop['시군구']))
print(len(ka), len(kv), len(kp))
print('사고에만:', sorted(ka - kp)); print('인구에만:', sorted(kp - ka)); print('차량에만:', sorted(kv - kp, key=str))

In [ ]:
import re
age_cols = [c for c in pop.columns if re.match(r'^\d+세', c)]
age = {c: int(re.match(r'^(\d+)', c).group(1)) for c in age_cols}
pop[age_cols] = pop[age_cols].apply(pd.to_numeric, errors='coerce')
pop['인구'] = pop[age_cols].sum(axis=1)
pop['65+'] = pop[[c for c in age_cols if age[c] >= 65]].sum(axis=1)
pop['70+'] = pop[[c for c in age_cols if age[c] >= 70]].sum(axis=1)
P = pop.groupby(['시도','시군구'])[['인구','65+','70+']].sum()

A = acc.groupby(['시도','시군구'])[['전체','고령']].sum()

vp = veh[(veh['회원구분명'] == '개인') & (veh['차종명'] != '특수')]
vp = vp.assign(고령차량=vp['대수'].where(vp['연령'].isin(['70대','80대','90대']), 0))
V = vp.groupby(['시도','시군구'])[['대수','고령차량']].sum().rename(columns={'대수':'개인차량'})

S = P.join(A).join(V)
S['고령화율'] = S['65+'] / S['인구']
S['고령사고_천명당'] = S['고령'] / 5 / S['65+'] * 1000
S['고령사고비중'] = S['고령'] / S['전체']
S['고령차량_백명당'] = S['고령차량'] / S['70+'] * 100
print(S.shape, S.isna().sum().sum())
print(S[['고령화율','고령사고_천명당','고령사고비중','고령차량_백명당']].describe().round(3))
print(S.sort_values('고령차량_백명당', ascending=False).head(8)[['고령화율','고령차량_백명당','고령사고_천명당']])

In [ ]:
pop = pop.copy()
S.to_csv('sgg_step1.csv', encoding='utf-8-sig')
cols = ['고령화율','고령사고_천명당','고령사고비중','고령차량_백명당']
print(S[cols].corr().round(2))
print(S.sort_values('고령차량_백명당').head(8)[['고령화율','고령차량_백명당','고령사고_천명당']])

In [ ]:
y = pd.read_csv('acc_year.csv')
t = y.groupby('사고발생년')[['size','sum']].sum(); t['비중'] = t['sum'] / t['size']
print(t)

In [ ]:
!apt-get -qq install fonts-nanum > /dev/null
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'; plt.rcParams['axes.unicode_minus'] = False

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.bar(t.index.astype(str), t['size'], color='#d9d9d9', label='전체 사고')
ax1.bar(t.index.astype(str), t['sum'], color='#e4572e', label='고령운전자 사고')
ax1.set_ylabel('사고 건수')
ax2 = ax1.twinx(); ax2.plot(t.index.astype(str), t['비중']*100, color='#222', marker='o', label='고령운전자 비중')
ax2.set_ylabel('비중(%)'); ax2.set_ylim(0, 25)
for x, v in zip(t.index.astype(str), t['비중']*100): ax2.annotate(f'{v:.1f}%', (x, v), textcoords='offset points', xytext=(0, 8), ha='center')
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9)); plt.title('전체 사고는 줄고, 고령운전자 사고는 늘었다 (2020–2024)')
plt.tight_layout(); plt.savefig('fig1_trend.png', dpi=200); plt.show()

In [ ]:
import geopandas as gpd
from shapely import wkt

fs = ['EmdBoundaryCapital.csv','EmdBoundaryGangwonChungcheong.csv','EmdBoundaryJeolla.csv','EmdBoundaryGyeongsang.csv']
b = pd.concat([pd.read_csv(f, dtype=str) for f in fs])
b = gpd.GeoDataFrame(b, geometry=b['wkt_epsg5186'].apply(wkt.loads), crs=5186).drop(columns='wkt_epsg5186')

n = b['adzn_nm'].str.split()                       # 예: '경기도 수원시 장안구 파장동'
b['시도'] = n.str[0].map(sd); b['시군구'] = n.str[1]   # 일반구는 시 단위로
b.loc[b['시도'] == '세종', '시군구'] = '세종시'
b = fx(b)
print(len(b), b['시도'].isna().sum())

g = b.dissolve(by=['시도','시군구'])[['geometry']]   # 시군구로 합치기 (1~2분)
g['geometry'] = g.simplify(100)                     # 100m 단위로 단순화해 가볍게
print(len(g))
print('지표에만:', set(S.index) - set(g.index)); print('경계에만:', set(g.index) - set(S.index))

In [ ]:
b['geometry'] = b.buffer(0)                         # 꼬인 도형 정리
print(b.is_valid.mean())

In [ ]:
g = b.dissolve(by=['시도','시군구'])[['geometry']]
g['geometry'] = g.simplify(100)
print(len(g))
print('지표에만:', set(S.index) - set(g.index)); print('경계에만:', set(g.index) - set(S.index))

In [ ]:
m = g.join(S)
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
for ax, col, title, cmap in [(axes[0], '고령차량_백명당', '70세 이상 100명당 고령자 명의 차량', 'Oranges'),
                              (axes[1], '고령사고_천명당', '65세 이상 1천 명당 고령운전자 사고 (연평균)', 'Reds')]:
    lo, hi = m[col].quantile([0.05, 0.95])
    m.plot(column=col, cmap=cmap, vmin=lo, vmax=hi, legend=True, ax=ax,
           edgecolor='white', linewidth=0.2, legend_kwds={'shrink': 0.6})
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.savefig('fig2_map_sgg.png', dpi=200); plt.show()

g.to_file('sgg.gpkg')   # 시군구 경계 저장 (다음에 재사용)

In [ ]:
S['차량_pct'] = S['고령차량_백명당'].rank(pct=True)
S['사고_pct'] = S['고령사고_천명당'].rank(pct=True)
both = S[(S['차량_pct'] >= 0.75) & (S['사고_pct'] >= 0.75)]
print(len(both), '개 시군구')
print(both[['고령화율', '고령차량_백명당', '고령사고_천명당']].round(2).sort_values('고령사고_천명당', ascending=False).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(S['고령차량_백명당'], S['고령사고_천명당'], s=12, c='lightgray')
ax.scatter(both['고령차량_백명당'], both['고령사고_천명당'], s=25, c='crimson')
for (sd_, sg), r in both.iterrows():
    ax.annotate(f'{sd_} {sg}', (r['고령차량_백명당'], r['고령사고_천명당']), fontsize=8, xytext=(3, 3), textcoords='offset points')
ax.axvline(S['고령차량_백명당'].quantile(0.75), ls='--', c='gray', lw=0.8)
ax.axhline(S['고령사고_천명당'].quantile(0.75), ls='--', c='gray', lw=0.8)
ax.set_xlabel('70세 이상 100명당 고령자 명의 차량')
ax.set_ylabel('65세 이상 1천 명당 고령운전자 사고 (연평균)')
ax.set_title('시군구별 고령 운전 의존도 × 사고 위험')
plt.tight_layout(); plt.savefig('fig3_scatter.png', dpi=200); plt.show()

In [ ]:
o = vp[vp['연령'].isin(['70대', '80대', '90대'])]
print((o.groupby('차종명')['대수'].sum() / o['대수'].sum()).round(3))
t = o.groupby(['시도', '시군구', '차종명'])['대수'].sum().unstack(fill_value=0)
t = t.div(t.sum(axis=1), axis=0).round(2)
print(t.loc[[('부산', '강서구'), ('대전', '유성구'), ('제주', '서귀포시'), ('경북', '성주군'), ('서울', '강남구')]])

In [ ]:
m = pd.read_csv('BjdHjdMap20250401.csv', dtype=str)
m = m[m['행정동코드'].str[:2].isin(['11', '28', '41'])]
m = m[~m['행정동코드'].str.endswith('00000') & ~m['법정동코드'].str.endswith('00000')]
m['emd_cd'] = m['법정동코드'].str[:8]
e = m[['행정동코드', 'emd_cd']].drop_duplicates()
p = {}
def root(x):
    while p.setdefault(x, x) != x:
        x = p[x]
    return x
for h, d in zip('H' + e['행정동코드'], 'B' + e['emd_cd']):
    p[root(h)] = root(d)
e['단위'] = [root('B' + x) for x in e['emd_cd']]
h2u = e.drop_duplicates('행정동코드').set_index('행정동코드')['단위']
b2u = e.drop_duplicates('emd_cd').set_index('emd_cd')['단위']
print('분석단위', e['단위'].nunique(), '| 행정동', e['행정동코드'].nunique(), '| 법정동', e['emd_cd'].nunique())

In [ ]:
pc = pop[pop['행정기관코드'].str[:2].isin(['11', '28', '41'])].copy()
fm = pd.read_csv('PopCodeFix.csv', dtype=str)
pc['행정동코드'] = pc['행정기관코드'].replace(dict(zip(fm['인구파일_행정기관코드'], fm['연계표_행정동코드'].str.split('|').str[0])))
pc['단위'] = pc['행정동코드'].map(h2u)
UP = pc.groupby('단위')[['인구', '65+', '70+']].sum()

bd = b[b['시도'].isin(['서울', '인천', '경기'])].copy()
bd['단위'] = bd['emd_cd'].map(b2u)
opo = b2u['41610250']   # 광주시 오포읍 → 2025년 6개 동으로 전환
bd.loc[bd['단위'].isna() & (bd['시군구'] == '광주시'), '단위'] = opo

ae = pd.read_csv('acc_emd.csv')
ae = ae[ae['사고발생시도'].isin(['서울', '인천', '경기'])].fillna(1)   # 비공개(0~2건)는 1건으로
ae['사고발생읍면동'] = ae['사고발생읍면동'].replace({'퇴계원면': '퇴계원읍', '능서면': '세종대왕면', '남사면': '남사읍'})
ae = ae.merge(bd[['시도', '시군구', 'emd_nm', '단위']], left_on=['사고발생시도', '사고발생시군구', '사고발생읍면동'], right_on=['시도', '시군구', 'emd_nm'], how='left')
ae.loc[(ae['사고발생시군구'] == '광주시') & (ae['사고발생읍면동'] == '오포읍'), '단위'] = opo
UA = ae.groupby('단위')[['전체', '고령']].sum()
print('인구 미연결', pc['단위'].isna().sum(), '| 경계 미연결', bd['단위'].isna().sum(), '| 사고 미연결(고령)', ae.loc[ae['단위'].isna(), '고령'].sum())

In [ ]:
nm = bd.groupby('단위').agg(시도=('시도', 'first'), 시군구=('시군구', 'first'), 동=('emd_nm', lambda s: '·'.join(s[:3]) + ('…' if len(s) > 3 else '')))
U = nm.join(UP).join(UA)
U[['전체', '고령']] = U[['전체', '고령']].fillna(0)
U['고령화율'] = U['65+'] / U['인구']
U['고령사고_천명당'] = U['고령'] / 5 / U['65+'] * 1000
U['소규모'] = U['65+'] < 500
U.to_csv('cap_unit.csv', encoding='utf-8-sig')

gu = bd.dissolve(by='단위')[['geometry']]
gu['geometry'] = gu.simplify(30)
mu = gu.join(U)
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
for a, col, cm, t in [(ax[0], '고령화율', 'Purples', '65세 이상 인구 비율'), (ax[1], '고령사고_천명당', 'Reds', '65세 이상 1천 명당 고령운전자 사고')]:
    mu.plot(column=col, cmap=cm, legend=True, ax=a, vmin=mu[col].quantile(.05), vmax=mu[col].quantile(.95), edgecolor='white', linewidth=0.1)
    a.set_title(t); a.axis('off')
plt.tight_layout(); plt.savefig('fig4_map_cap.png', dpi=200); plt.show()
print(U[~U['소규모']].sort_values('고령사고_천명당', ascending=False).head(8)[['시군구', '동', '65+', '고령', '고령사고_천명당']].to_string())

In [ ]:
er = pd.read_csv('ersr_sgg.csv')
er['연'] = er['파일'].str[-10:-6].astype(int)
er = er[er['연'].between(2018, 2025)]
er['고령'] = er['연령'].isin(['70대', '80대', '90대'])
er['폐차'] = er['신청구분명'].str.contains('폐차|차령초과')
t = er.groupby(['연', '고령'])['건수'].sum().unstack()
t.columns = ['70세 미만', '70세 이상']
t['고령비중'] = t['70세 이상'] / t.sum(axis=1)
p = er[er['폐차']].groupby(['연', '고령'])['건수'].sum().unstack()
t['폐차_고령비중'] = p[True] / p.sum(axis=1)
print(t.round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(t.index, t['70세 이상'] / 1e4, color='#e0915a')
ax.set_ylabel('70세 이상 말소 (만 대)')
ax2 = ax.twinx()
ax2.plot(t.index, t['폐차_고령비중'] * 100, 'o-', c='#333')
ax2.set_ylabel('폐차 중 70세 이상 비중 (%)')
ax.set_title('고령자 명의 차량 말소 추이 (개인, 2018~2025)')
plt.tight_layout(); plt.savefig('fig5_ersr.png', dpi=200); plt.show()

In [ ]:
S['위험'] = S['고령사고_천명당'].rank(pct=True)
S['의존'] = S['고령차량_백명당'].rank(pct=True)
S['지수_임시'] = S[['위험', '의존']].mean(axis=1)
print(S.sort_values('지수_임시', ascending=False).head(15)[['고령화율', '고령차량_백명당', '고령사고_천명당', '지수_임시']].round(2).to_string())
S.to_csv('sgg_step1.csv', encoding='utf-8-sig')